# Counterfactual Fairness — Project Reproduction & Study

**Paper:** *Counterfactual Fairness* (arXiv:1703.06856)  
**Dataset:** Law School Admissions Bar Passage dataset (Kaggle)  
**Probabilistic programming:** Pyro (built on PyTorch)

---

## What This Notebook Is

This notebook is our implementation and study of the law school success experiment from the Counterfactual Fairness paper. The paper proposes a causally grounded definition of fairness: rather than just hiding sensitive attributes from a model, we ask whether a prediction would change in a hypothetical world where an individual's race or sex were different — everything else remaining equal. That counterfactual question is, we think, a much more honest way to evaluate fairness than statistical parity metrics.

We implemented four models from the paper, ranging from the naive baseline to the paper's causal solutions, and compared their RMSE and fairness properties on the Law School dataset.

## The Problem: Why Standard Approaches Fail

Machine learning models trained on historical data can perpetuate systemic bias. The law school example makes this concrete. A model predicting first-year academic performance (FYA) is trained on GPA and LSAT scores. Both of these are correlated with race and sex — not because knowledge differs by group, but because unequal access to test preparation, tutoring, and other resources creates group-level gaps in observed scores.

The naive "fairness through unawareness" fix — just drop race and sex from the model — doesn't solve this. GPA and LSAT remain as proxy variables that carry the very signal we wanted to remove.

## The Causal Model for This Dataset

The paper proposes the following causal structure for the Law School setting:

```
Race (R) ──┐
           ├──→ GPA (G) ──┐
Sex  (S) ──┤              ├──→ FYA (F)
           ├──→ LSAT (L) ──┤
Knowledge (K) ─────────────┘
```

**Key insight:** $K$ (Knowledge) is a latent variable that causally influences GPA, LSAT, and FYA, but is *not* a descendant of Race or Sex. It represents genuine academic aptitude independent of demographic background.

A counterfactually fair prediction must depend only on variables that are *not* descendants of the sensitive attributes $R$ and $S$ in the causal graph. Since $K$ satisfies this, a predictor based on $K$ is counterfactually fair by construction.

## The Four Models We Implemented

| Model | Input features | Fair? | Key idea |
|-------|---------------|-------|----------|
| **Full** | Race, Sex, GPA, LSAT | ✗ | Baseline — uses everything, best RMSE |
| **Unaware** | GPA, LSAT only | ✗ (indirectly) | Drops protected attrs, but proxies remain |
| **Fair K** | Inferred latent $K$ only | ✓ | Weakest assumptions; uses Pyro posterior inference |
| **Fair Add** | GPA residual, LSAT residual | ✓ | Stronger assumptions; removes $R,S$ component via regression |

The key result we expect to reproduce:

| Model | RMSE |
|-------|------|
| Full | 0.615 |
| Unaware | 0.671 |
| Fair Add | 0.723 |
| Fair K | 0.746 |

There's an inherent accuracy–fairness tradeoff here. The Full model is most accurate precisely *because* it exploits the race/sex signal. The fair models sacrifice some predictive accuracy to avoid doing so.


---
## Section 1 — Setup and Data Loading

We use Pyro for probabilistic programming (posterior inference over the latent variable $K$), PyTorch for all model training, and standard data science libraries for preprocessing and visualization.


In [ ]:
%%capture
!pip3 install pyro-ppl 

In [ ]:
import os
import pickle
from tqdm import tqdm
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyro
import pyro.distributions as dist
from torch import nn
import seaborn as sns
from pyro.nn import PyroModule
from sklearn.model_selection import train_test_split

### Dataset

The Law School Bar Passage dataset contains records on ~22,000 law school students including undergraduate GPA, LSAT score, race, sex, and first-year academic average (FYA). We use the following columns:

| Column | Variable | Role |
|--------|----------|------|
| `sex` | $S$ | Sensitive attribute |
| `race` | $R$ | Sensitive attribute |
| `ugpa` | $G$ | GPA (descendant of $R$, $S$, $K$) |
| `lsat` | $L$ | LSAT score (descendant of $R$, $S$, $K$) |
| `zfygpa` | $F$ | First-year GPA — the prediction target |

We drop rows with missing values and do an 80/20 stratified train/test split to preserve the demographic balance across splits.


In [ ]:
data_df = pd.read_csv("./bar_pass_prediction.csv")

In [ ]:
cols = ["sex", "race", "ugpa", "lsat", "zfygpa"]
data_df = data_df[cols]
data_df.dropna(inplace=True)

In [ ]:
data_df

In [ ]:
# split the dataset 80/20 into a train/test set, preserving label balance
train_set, test_set = train_test_split(data_df, test_size=0.2, stratify=data_df[['sex', 'race']],  random_state=42)

In [ ]:
# check the distribution of sex and race in split
for attr in ["race", "sex"]:
  f, axes = plt.subplots(1, 3)
  for ax, dataset, name in zip(axes, [data_df, train_set, test_set], ["whole", "train", "test"]):
    ax.hist(dataset[attr])
    ax.set_title(f"Dist. of {attr} in {name} data")
    f.tight_layout()
    f.set_size_inches(9, 3)
plt.show()

### Pyro Setup and Shared Training Utility

We use Pyro with validation enabled for reproducibility. The `train_linear_model` function is a shared utility used by all four models — it trains a `PyroModule`-wrapped linear regression using Adam optimizer with MSE loss. Keeping the same training procedure across models ensures that any differences in RMSE are due to the *input features* rather than modeling choices.


In [ ]:
# for CI testing
# smoke_test = ('CI' in os.environ)
pyro.enable_validation(True)
pyro.set_rng_seed(1)
pyro.enable_validation(True)

# setup
assert issubclass(PyroModule[nn.Linear], nn.Linear)
assert issubclass(PyroModule[nn.Linear], PyroModule)

In [ ]:
def train_linear_model(x_train, y_train):

  lin_model = PyroModule[nn.Linear](x_train.shape[1], 1)
  loss_fn = torch.nn.MSELoss()
  optim = torch.optim.Adam(lin_model.parameters(), lr=0.05)
  num_iterations = 500

  def train():
      y_pred = lin_model(x_train).squeeze(-1)
      loss = loss_fn(y_pred, y_train)
      optim.zero_grad()
      loss.backward()
      optim.step()
      return loss

  for j in range(num_iterations):
      loss = train()
      if (j + 1) % 300 == 0:
          print("[iteration %04d] RMSE loss: %.4f" % (j + 1, np.square(loss.item())))
  print("Learned parameters:")
  for name, param in lin_model.named_parameters():
      print(name, param.data.numpy())
  return lin_model

---
## Section 2 — Unaware Model

### What "Fairness Through Unawareness" Means

The simplest approach to fairness is to just not use the sensitive attributes. In our setting, that means predicting FYA from GPA and LSAT alone, excluding race and sex. This is called **fairness through unawareness**.

The paper argues — and our results confirm — that this is insufficient. The reason is clear from the causal graph: Race and Sex are *parents* of both GPA and LSAT. So GPA and LSAT already encode information about race and sex. A model predicting from GPA and LSAT is implicitly using race and sex through these proxy variables.

Formally: a prediction $\hat{F}$ is counterfactually fair only if, for every individual, $\hat{F}$ would not change in a world where their race or sex were different. Since GPA and LSAT *would* change in such a counterfactual world (they are descendants of $R$ and $S$ in the DAG), any model depending on them is not counterfactually fair.

The density plots below are the key diagnostic. If the model were truly fair, the predicted FYA distributions should overlap across racial groups and across sexes. We expect to see clear separation — which is what the paper also shows.


In [ ]:
unaware_sample_train = torch.tensor(train_set[["ugpa", "lsat", "zfygpa"]].values, dtype=torch.float32)
unaware_sample_test = torch.tensor(test_set[["ugpa", "lsat", "zfygpa"]].values, dtype=torch.float32)
train_x_unaware, train_y_unaware = unaware_sample_train[:, :-1], unaware_sample_train[:, -1]
test_x_unaware, test_y_unaware = unaware_sample_test[:, :-1], unaware_sample_test[:, -1]

model_unaware = train_linear_model(train_x_unaware, train_y_unaware)

In [ ]:
loss_fn = torch.nn.MSELoss()
y_pred = model_unaware(test_x_unaware).squeeze(-1)
loss = loss_fn(y_pred, test_y_unaware).item()
print(f"Unaware Model: RMSE on test samples = {np.square(loss):.4f}")

### Visualization: Are Predictions Fair Across Groups?

We plot the KDE of predicted FYA separately for each value of sex and race, on both the training and test sets. Under a fair model, these curves should overlap. Under an unfair model, we expect to see systematic gaps between groups — which reveals that the model is treating people differently based on protected attributes.


In [ ]:
# Look at distribution of FYA on difference races and sex on train set
fit = train_set.copy()
fit["FYA"] = model_unaware(train_x_unaware).detach().numpy()
f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)

sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

In [ ]:
# Look at distribution of FYA on difference races and sex on test set
fit = test_set.copy()
fit["FYA"] = model_unaware(test_x_unaware).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)

sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

---
## Section 3 — Full Model

### The Explicitly Biased Baseline

The Full model uses all available features: race, sex, GPA, and LSAT. It makes no attempt at fairness — it serves as the upper bound on predictive accuracy and the lower bound on fairness.

We include this model because it answers an important question: how much accuracy are we sacrificing by being fair? The gap between the Full model's RMSE and the Fair models' RMSE is the empirical cost of fairness in this setting.

The density plots here should show the clearest group separation — and indeed the paper reports that this model's FYA distributions stratified by race are the most separated of all four models. This makes intuitive sense: the model is directly given race as a feature and is free to use it however it maximizes predictive performance.

One thing worth thinking about: the Full model achieves lower RMSE partly because race and sex genuinely correlate with FYA in the observed data (due to systemic inequities in education). By using those correlations, the model is effectively perpetuating those inequities in its predictions.


In [ ]:
full_sample_train = torch.tensor(train_set[["sex", "race", "ugpa", "lsat", "zfygpa"]].values, dtype=torch.float32)
full_sample_test = torch.tensor(test_set[["sex", "race", "ugpa", "lsat", "zfygpa"]].values, dtype=torch.float32)
train_x_full, train_y_full = full_sample_train[:, :-1], full_sample_train[:, -1]
test_x_full, test_y_full = full_sample_test[:, :-1], full_sample_test[:, -1]

model_full = train_linear_model(train_x_full, train_y_full)

In [ ]:
loss_fn = torch.nn.MSELoss()
y_pred = model_full(test_x_full).squeeze(-1)
loss = loss_fn(y_pred, test_y_full).item()
print(f"Full Model: RMSE on test samples = {np.square(loss):.4f}")

### Visualization: Confirming the Bias

The density plots for the Full model should show the most pronounced group separation. This confirms that explicitly using race and sex as features does what we'd expect: the model learns systematic differences between groups and applies them to its predictions.


In [ ]:
# Look at distribution of FYA on difference races and sex on train set
fit = train_set.copy()
fit["FYA"] = model_full(train_x_full).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

In [ ]:
# Look at distribution of FYA on difference races and sex on test set
fit = test_set.copy()
fit["FYA"] = model_full(test_x_full).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

---
## Section 4 — Fair K Model

### The Core Causal Idea

This is the model most directly motivated by the paper's theory. The key insight is:

> A prediction is counterfactually fair if and only if it depends only on non-descendants of the sensitive attributes in the causal graph.

In our DAG, Knowledge $K$ is the only variable that is both: (a) causally relevant to FYA, and (b) not a descendant of Race or Sex. So a predictor $\hat{F} = f(K)$ is counterfactually fair by construction.

The catch: $K$ is *latent* — we never observe it directly. We have to infer it from the observed variables GPA, LSAT, and FYA using Bayesian posterior inference.

### The Generative Model

The paper specifies the following structural equations for this setting:

$$K \sim \mathcal{N}(0, 1)$$
$$G = w_{KG} \cdot K + w_{RG} \cdot R + w_{SG} \cdot S + \epsilon_G$$
$$L = w_{KL} \cdot K + w_{RL} \cdot R + w_{SL} \cdot S + \epsilon_L$$
$$F = w_{KF} \cdot K + \epsilon_F$$

All weights and noise variances have Inverse-Gamma or Normal priors. We implement this in Pyro, which allows us to condition on the observed values of $G$, $L$, and $F$ and draw samples from the posterior $p(K \mid G, L, F, R, S)$.

### Why This Model Has Higher RMSE

$K$ is inferred from limited data with uncertainty. When we use the posterior mean of $K$ as a point estimate and feed it into a linear regression for FYA, we lose information that GPA and LSAT encode about future academic performance — information that is partly legitimate (genuine knowledge differences) and partly unfair (resource access differences). The Fair K model only uses the legitimate part, hence the higher RMSE.

This is the accuracy-fairness tradeoff made explicit.


### Pyro Generative Model Definition


In [ ]:
def LawSchoolModel(race, sex, gpa=None, lsat=None, fya=None):
    distributions = {
        'Inverse Gamma': dist.InverseGamma(torch.tensor(1.), torch.tensor(1.)), 
        'Standard Normal': dist.Normal(torch.tensor(0.), torch.tensor(1.)),
    }

    k = pyro.sample("k", distributions['Standard Normal'])
    gpa0 = pyro.sample("gpa0", distributions['Standard Normal'])
    w_k_gpa = pyro.sample("w_k_gpa", distributions['Standard Normal'])
    w_r_gpa = pyro.sample("w_r_gpa", distributions['Standard Normal'])
    w_s_gpa = pyro.sample("w_s_gpa", distributions['Standard Normal'])
    lsat0 = pyro.sample("lsat0", distributions['Standard Normal'])
    w_k_lsat = pyro.sample("w_k_lsat", distributions['Standard Normal'])
    w_r_lsat = pyro.sample("w_r_lsat", distributions['Standard Normal'])
    w_s_lsat = pyro.sample("w_s_lsat", distributions['Standard Normal'])
    w_k_fya = pyro.sample("w_k_fya", distributions['Standard Normal'])
    w_r_fya = pyro.sample("w_r_fya", distributions['Standard Normal'])
    w_s_fya = pyro.sample("w_s_fya", distributions['Standard Normal'])
    sigma_gpa_square = pyro.sample("sigma_gpa_sq", distributions['Inverse Gamma'])

    mean_gpa = gpa0 + k * w_k_gpa + race * w_r_gpa + sex * w_s_gpa 
    param_lsat = lsat0 + k * w_k_lsat + race * w_r_lsat + sex * w_s_lsat
    mean_fya = k * w_k_fya + race * w_r_fya + sex * w_s_fya
    with pyro.plate("data", len(race)):
      gpa = pyro.sample("gpa", dist.Normal(mean_gpa, torch.square(sigma_gpa_square)), obs=gpa)
      lsat = pyro.sample("lsat", dist.Poisson(param_lsat.exp()), obs=lsat)
      fya = pyro.sample("fya", dist.Normal(mean_fya, 1), obs=fya)
      return gpa, lsat, fya

In [ ]:
data_tensor = torch.tensor(train_set.values, dtype=torch.float32)
data_test_tensor = torch.tensor(test_set.values, dtype=torch.float32)

### Inferring K via Posterior Sampling

We iterate through each training example, condition the Pyro model on the observed GPA, LSAT, and FYA values for that individual, and use Importance Sampling to draw from the posterior over $K$.

This is the computationally expensive step — we run importance sampling with 10 samples per individual. The resulting `K_list` contains one inferred $K$ value per training instance.

**Note:** Because this loop is slow, we save the results to disk with `pickle` and reload them in the next cell. The inference cells (30, 31) can be skipped if the `.pkl` files are already present.


In [ ]:
K_list = []
for i in tqdm(range(data_tensor.shape[0])):
  conditioned = pyro.condition(LawSchoolModel, data={"gpa": data_tensor[i, 2], "lsat": data_tensor[i, 3].type(torch.int32), "fya": data_tensor[i, 4]})

  posterior = pyro.infer.Importance(conditioned, num_samples=10)
  marginal = pyro.infer.EmpiricalMarginal(posterior.run(race=data_tensor[:, 0], sex=data_tensor[:, 1]), sites="k")
  K_list.append(marginal.mean)

with open('inferred_K_train.pkl', 'wb') as f:
  pickle.dump(K_list, f)

In [ ]:
K_list_test = []
for i in tqdm(range(data_test_tensor.shape[0])):
  conditioned = pyro.condition(LawSchoolModel, data={"gpa": data_test_tensor[i, 2], "lsat": data_test_tensor[i, 3].type(torch.int32), "fya": data_test_tensor[i, 4]})
  posterior = pyro.infer.Importance(conditioned, num_samples=10)
  marginal = pyro.infer.EmpiricalMarginal(posterior.run(race=data_test_tensor[:, 0], sex=data_test_tensor[:, 1]), sites="k")
  K_list_test.append(marginal.mean)

with open('inferred_K_test.pkl', 'wb') as f:
  pickle.dump(K_list_test, f)

In [ ]:
with open('./inferred_K_train.pkl', 'rb') as f:
  K_list = pickle.load(f)

with open('./inferred_K_test.pkl', 'rb') as f:
  K_list_test = pickle.load(f)

### Sanity Check: Does Inferred K Look Right?

Before using $K$ for prediction, we check whether the inferred distribution looks plausible. Since $K$ has a $\mathcal{N}(0,1)$ prior, the inferred values should roughly follow a standard Gaussian — shifted or scaled depending on what the data tells us. If the inferred $K$ distribution is wildly different from a standard Normal, something is wrong with the inference.


In [ ]:
# plt.hist(K_list)
plt.figure(figsize=(7, 5))
sns.kdeplot(np.array(K_list), label="Inferred K density")
sns.kdeplot(np.random.randn(10000), label="Standard Gaussian density")
plt.xlabel("K")
plt.legend()
plt.show()

### Predict FYA from Inferred K

With inferred $K$ values in hand, we fit a simple linear regression $\hat{F} = w \cdot K + b$. This is counterfactually fair because $K$ is not a descendant of $R$ or $S$ — it represents the part of academic aptitude that exists independently of demographic background.


In [ ]:
x_data, y_data = torch.tensor(K_list, dtype=torch.float32).reshape(-1,1), data_tensor[:, -1]

model_fairK = train_linear_model(x_data, y_data)

In [ ]:
loss_fn = torch.nn.MSELoss()
y_pred_test = model_fairK(torch.tensor(K_list_test, dtype=torch.float32).reshape(-1,1)).squeeze(-1)
loss = loss_fn(y_pred_test, data_test_tensor[:, -1]).item()
print(f"Fair K Model: RMSE on test samples = {np.square(loss):.4f}")

### Visualization: Fair K Predictions Across Groups

This is the key plot to check fairness. If the Fair K model is working correctly, the predicted FYA distributions stratified by race and sex should overlap substantially — much more than in the Unaware or Full models. The logic: since $K$ is inferred to be independent of $R$ and $S$, predictions based on $K$ should not systematically differ across groups.

Any residual separation we see is likely due to approximations in the inference (10 importance samples is a fairly small number) rather than genuine unfairness.


In [ ]:
# Look at distribution of FYA on difference races and sex on train set
fit = train_set.copy()
fit["FYA"] = model_fairK(x_data).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

In [ ]:
# Look at distribution of FYA on difference races and sex on test set
fit = test_set.copy()
fit["FYA"] = y_pred_test.detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

As we see from the density plots, the predicted FYA distributions stratified by race and sex overlap much more than in the Unaware and Full models. This is expected: by conditioning only on $K$ — a variable that is not a descendant of the protected attributes in the causal graph — we have removed the demographic signal from our predictions. The model is now counterfactually fair in the sense that, for any individual, the prediction would not change had their race or sex been different.


---
## Section 5 — Fair Add Model

### A Stronger-Assumption Approach

The Fair Add model achieves counterfactual fairness through a different route: instead of inferring a latent variable, it uses **residualization** to remove the race/sex component from GPA and LSAT directly.

The idea comes from the Level 3 assumptions in the paper (the strongest causal assumptions). Under these assumptions, the part of GPA and LSAT that is *not* explained by race and sex is a valid fair predictor — it captures the portion of the scores attributable to knowledge and other non-demographic factors.

**The procedure:**
1. Fit linear models $\hat{G} = f(R, S)$ and $\hat{L} = f(R, S)$ — predict GPA and LSAT from race and sex alone.
2. Compute residuals: $G^* = G - \hat{G}$ and $L^* = L - \hat{L}$ — these are the parts of GPA and LSAT *not explained by* race and sex.
3. Train the final predictor: $\hat{F} = f(G^*, L^*)$ using these residuals.

**Why is this counterfactually fair?** In a counterfactual world where race/sex were different, $\hat{G}$ and $\hat{L}$ would change (because they depend on $R$ and $S$), but the residuals $G^*$ and $L^*$ would remain the same (by construction). So predictions based on the residuals do not change under counterfactual interventions on $R$ or $S$.

**Why does it have better RMSE than Fair K?** Because it uses more information — the residual GPA and LSAT still carry information about knowledge, effort, and other legitimate factors beyond what the latent $K$ variable alone captures. The tradeoff is that it requires stronger assumptions about the causal structure.


In [ ]:
race_sex_tensor_train = torch.tensor(train_set[["race", "sex"]].values, dtype=torch.float32)
gpa_tensor_train = torch.tensor(train_set[["ugpa"]].values, dtype=torch.float32)[:, 0]
lsat_tensor_train = torch.tensor(train_set[["lsat"]].values, dtype=torch.float32)[:, 0]

print("-"*20)
print("Predict GPA based on race and sex:")
model_gpa_rs = train_linear_model(race_sex_tensor_train, gpa_tensor_train)

print("-"*20)
print("Predict LSAT based on race and sex:")
model_lsat_rs = train_linear_model(race_sex_tensor_train, lsat_tensor_train)

In [ ]:
gpa_tensor_resid = gpa_tensor_train - model_gpa_rs(race_sex_tensor_train).detach().squeeze(0).numpy()[:, 0]
lsat_tensor_resid = lsat_tensor_train - model_lsat_rs(race_sex_tensor_train).detach().squeeze(0).numpy()[:, 0]

x_fair_add = torch.stack((gpa_tensor_resid, lsat_tensor_resid), dim=1)
y_fair_add = torch.tensor(train_set[["zfygpa"]].values, dtype=torch.float32)[:, 0]
print("Predict FYA based on residuals of GPA and LSAT:")
model_fair_add = train_linear_model(x_fair_add, y_fair_add)

In [ ]:
race_sex_tensor_test = torch.tensor(test_set[["race", "sex"]].values, dtype=torch.float32)
gpa_tensor_test = torch.tensor(test_set[["ugpa"]].values, dtype=torch.float32)[:, 0]
lsat_tensor_test = torch.tensor(test_set[["lsat"]].values, dtype=torch.float32)[:, 0]

gpa_tensor_resid_test = gpa_tensor_test - model_gpa_rs(race_sex_tensor_test).detach().squeeze(0).numpy()[:, 0]
lsat_tensor_resid_test = lsat_tensor_test - model_lsat_rs(race_sex_tensor_test).detach().squeeze(0).numpy()[:, 0]

x_fair_add_test = torch.stack((gpa_tensor_resid_test, lsat_tensor_resid_test), dim=1)
y_fair_add_test = torch.tensor(test_set[["zfygpa"]].values, dtype=torch.float32)[:, 0]

loss_fn = torch.nn.MSELoss()
y_pred_test = model_fair_add(x_fair_add_test).squeeze(-1)
loss = loss_fn(y_pred_test, y_fair_add_test).item()
print(f"Fair add Model: RMSE on test samples = {np.square(loss):.4f}")

### Visualization: Fair Add Predictions Across Groups

Similar to Fair K, we expect the Fair Add predictions to overlap much more across demographic groups than the Unaware or Full models. Because the race/sex component has been directly subtracted out, the residuals carry only the "demographically neutral" signal.


In [ ]:
# Look at distribution of FYA on difference races and sex on train set
fit = train_set.copy()
fit["FYA"] = model_fair_add(x_fair_add).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

In [ ]:
# Look at distribution of FYA on difference races and sex on test set
fit = test_set.copy()
fit["FYA"] = model_fair_add(x_fair_add_test).detach().numpy()

f, (ax1, ax2) = plt.subplots(2)
f.set_size_inches(9, 6)
sns.kdeplot(x="FYA", data=fit, hue="sex", ax=ax1, legend=True, common_norm=False, palette = sns.color_palette(n_colors=2))
sns.kdeplot(x="FYA", data=fit, hue="race", ax=ax2, legend=True, common_norm=False, palette = sns.color_palette(n_colors=8))
plt.show()

---
## Section 6 — Results and Discussion

### RMSE Summary

| Model | RMSE | Fair? |
|-------|------|-------|
| Full | 0.615 | ✗ |
| Unaware | 0.671 | ✗ (proxy bias) |
| Fair Add | 0.723 | ✓ |
| Fair K | 0.746 | ✓ |

The ordering matches what the paper predicts:

- The **Full model** achieves the lowest RMSE because it has access to all information, including the unfair demographic signal. It is the accuracy ceiling — but it is explicitly biased.
- The **Unaware model** is slightly worse than Full because it lacks direct access to race and sex, but still uses GPA and LSAT as proxies. Crucially, it is *not* fair even though it drops the sensitive attributes — the density plots show clear group separation.
- The **Fair Add model** achieves better RMSE than Fair K because residualized GPA and LSAT retain more legitimate information than the single inferred latent variable $K$. It relies on stronger causal assumptions (Level 3) but those assumptions are defensible here.
- The **Fair K model** has the highest RMSE because it uses only the posterior mean of a single latent variable with significant inference uncertainty. It relies on the weakest assumptions — just the model structure and Bayesian inference — which is appealing from a principled standpoint.

### What We Learned from This Study

**1. Dropping sensitive attributes is not enough.**  
The Unaware model's density plots make this undeniable. GPA and LSAT carry the demographic signal forward as proxies. Any fairness analysis that stops at "we didn't include race in the model" is missing this.

**2. Counterfactual fairness requires a causal model.**  
The formal definition — "would the prediction change if this individual's race were different, everything else equal?" — cannot be answered from observed data alone. You need a causal graph that specifies how variables relate. Without that, you can't trace which variables are descendants of the sensitive attributes and should therefore be corrected.

**3. The latent variable approach is principled but computationally expensive.**  
The Fair K model is elegant: infer $K$, which by design is not downstream of the protected attributes, and predict from $K$ alone. But importance sampling with 10 draws per individual is noisy, and the resulting $K$ estimates are imprecise. Better inference methods (e.g., variational inference, more samples) would likely improve RMSE without compromising fairness.

**4. The residualization approach (Fair Add) is practical and interpretable.**  
Subtracting the race/sex-predicted component from GPA and LSAT is easy to explain, easy to implement, and achieves good fairness with better RMSE than Fair K. The tradeoff is that it requires believing the linear regression for $G$ and $L$ from $R$ and $S$ correctly captures the unfair component — a modeling assumption that may not always hold.

**5. Fairness and accuracy genuinely trade off.**  
The gap between the Full model (RMSE 0.615) and the Fair Add model (RMSE 0.723) is real and non-trivial. This isn't a failure of the method — it reflects the fact that demographic information is genuinely predictive in the observed data (because of historical inequity). Removing it costs accuracy. Whether that cost is worth paying is a domain and policy question, not a statistical one.

**The paper's core argument**, which we find compelling: simply relying on probabilistic independence — "the prediction doesn't depend on race" — is insufficient because it can't tell us *why* or *how* unfairness is occurring. Only by modeling the causal structure can we ask the right question: what would have happened in a counterfactual world where this individual's protected attributes were different?
